# The Standard ReAct Agent in LangGraph

**ReAct** = *Reason* (LLM thinks / decides) + *Act* (a tool runs) in a loop, until the LLM decides it has enough info to answer.

This notebook builds it the "manual" way first (so every piece is visible), using the **standard, idiomatic LangGraph building blocks** — not custom code where a prebuilt exists. Follow the steps in order; each step names the one correct way to do that piece.

In [ ]:
import os
from typing import Annotated, TypedDict

from dotenv import load_dotenv
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch
from langgraph.graph import StateGraph, START, END, add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()


## Step 1 - State

**Standard:** one key, `messages`, typed as `Annotated[list[BaseMessage], add_messages]`.

- `add_messages` is the reducer that **appends** new messages instead of overwriting the list - without it every node return would wipe history.
- Don't add extra state keys (`tool_output`, `last_action`, etc.) unless you truly need them elsewhere. The message list already carries AI messages, tool calls, and `ToolMessage` results - that's the whole agent's memory.

In [ ]:
class State(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


## Step 2 - Tools

**Standard:** `@tool` decorator, with a type-hinted signature and a clear docstring.

- The **docstring is the only thing the LLM sees** to decide when/how to call the tool - write it like documentation for a person, not a code comment.
- Keep each tool doing one thing. Return plain strings/JSON-serializable data - that's what becomes the `ToolMessage` content.

In [ ]:
@tool
def get_word_length(word: str) -> int:
    """Return the number of characters in a word. Use this when the user asks how long a word is."""
    return len(word)


search_tool = TavilySearch(max_results=3, topic="general")

tools = [get_word_length, search_tool]


## Step 3 - LLM + `bind_tools`

**Standard:** create the LLM once at module scope, call `.bind_tools(tools)` once, reuse the bound model everywhere.

- Don't rebuild or re-bind inside the node function - it's wasted work on every single node call.
- Use `temperature=0` (or low) for agents. Tool-calling needs to be deterministic and well-formatted, not creative.

In [ ]:
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)
llm_with_tools = llm.bind_tools(tools)


## Step 4 - System prompt

**Standard:** one `SystemMessage`, set once, put in as the first message of the conversation (not re-sent by the node every turn).

- State the agent's role and, if relevant, "use tools when you need current/precise information; otherwise answer directly." The LLM already knows how to call tools once bound - you're steering *judgment*, not syntax.

In [ ]:
system_message = SystemMessage(
    content="You are a helpful assistant. Use the available tools when you need "
            "information you don't already know or that must be current/precise. "
            "Otherwise, answer directly."
)


## Step 5 - Agent node

**Standard:** a plain function `(state) -> dict`, invoke the bound LLM on `state["messages"]`, return only the **new** message(s).

- Return `{"messages": [response]}`, never the full rebuilt list - the `add_messages` reducer handles appending. Returning the full list is the #1 source of duplicated-history bugs.

In [ ]:
def agent_node(state: State) -> dict:
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}


## Step 6 - Tool execution node

**Standard:** `langgraph.prebuilt.ToolNode(tools)`. Don't hand-write a node that loops over `tool_calls` and invokes them yourself.

`ToolNode` already does the correct thing:
- runs **all** parallel tool calls the LLM made in one AI message
- matches each result back to its `tool_call_id` as a `ToolMessage`
- catches tool exceptions and turns them into an error `ToolMessage` instead of crashing the graph

In [ ]:
tool_node = ToolNode(tools)


## Step 7 - Routing (the "Re" in ReAct)

**Standard:** `langgraph.prebuilt.tools_condition`. Don't hand-write `should_continue` - this is the exact same check (`bool(last_message.tool_calls)`), maintained by LangGraph, and it returns the conventional labels `"tools"` / `"__end__"` that wire up directly.

In [ ]:
graph = StateGraph(State)

graph.add_node("agent", agent_node)
graph.add_node("tools", tool_node)

graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", tools_condition)  # routes to "tools" or END automatically
graph.add_edge("tools", "agent")  # <-- this edge is what makes it a loop, not a single tool call


## Step 8 - Memory / checkpointer

**Standard:**
- Dev / notebooks / tests -> `InMemorySaver()`
- Anything that must survive a process restart -> `SqliteSaver` or `PostgresSaver` (same interface, drop-in swap later)

Always compile **with** a checkpointer and always pass a `thread_id` in config - that's what gives the agent multi-turn memory and is a prerequisite for interrupts/human-in-the-loop later.

In [ ]:
checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer=checkpointer)
workflow


## Step 9 - Invoking: recursion limit

**Standard:** always pass `recursion_limit` in config for a tool-calling loop. If a tool keeps returning something the LLM keeps reacting to, an unbounded loop is a real failure mode, not a hypothetical - cap it explicitly instead of relying on the default (25).

In [ ]:
config = {"configurable": {"thread_id": "thread_1"}, "recursion_limit": 15}

result = workflow.invoke(
    {"messages": [system_message, HumanMessage(content="How many letters are in 'LangGraph', and who won the 2024 F1 championship?")]},
    config=config,
)
for m in result["messages"]:
    m.pretty_print()


## Bonus - the one-line shortcut

`langgraph.prebuilt.create_react_agent` builds **exactly the graph above** (agent node + `ToolNode` + `tools_condition` + loop edge) for you.

**When to use which:**
- **Use `create_react_agent`** for a standard tool-calling agent with no extra nodes - it's the same graph, less code to maintain.
- **Build it manually (as above)** the moment you need something between/around the standard nodes: a guardrail/moderation node, human-in-the-loop approval before a tool runs, message trimming/summarization, multiple LLMs, custom routing beyond "did it call a tool." The manual version is what you extend from.

In [ ]:
from langgraph.prebuilt import create_react_agent

quick_agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=system_message.content,
    checkpointer=InMemorySaver(),
)
